# **Project Name**- Flipkart Customer Service Satisfaction (ML)



##### **Project Type**    - Classification
##### **Contribution**    - Individual

# **Project Summary -**

In the fiercely competitive e-commerce industry, superior customer service is a critical differentiator for growth and loyalty. **Flipkart**, one of India’s largest e-commerce platforms, aims to enhance customer satisfaction by analyzing support interactions across multiple channels. This project focuses on understanding the drivers of customer satisfaction, evaluating the performance of customer service teams, and identifying actionable strategies to improve service delivery.

By uncovering key factors influencing satisfaction, Flipkart can:
- Optimize agent performance
- Resolve issues more efficiently
- Tailor support strategies to meet diverse customer needs

This ultimately enhances metrics like **CSAT** (Customer Satisfaction Score), fosters stronger customer retention, and strengthens brand loyalty.

---

## Dataset Description

The dataset contains detailed records of customer interactions, feedback, and satisfaction scores across various Flipkart support channels. It includes the following variables:

- **Customer demographics** (if available or anonymized)
- **Interaction timestamps** and **channels** (e.g., chat, phone, email)
- **Issue types** and **resolutions**
- **Customer satisfaction scores** (CSAT)
- **Agent performance metrics**

---

## Project Architecture & Tools

To effectively analyze and derive insights from the data, the following tools and libraries will be used:

- **Pandas & NumPy**: Data cleaning, transformation, and numerical operations for handling large datasets efficiently.
- **Matplotlib & Seaborn**: Visual exploration of trends, correlations, and patterns in customer interactions and satisfaction.
- **Scikit-learn**: Machine learning classification algorithms to predict customer satisfaction and evaluate influencing factors.
- **Faker (Optional)**: Generation of synthetic data or anonymization of sensitive customer information to protect privacy.

---

## Key Objectives

The primary objectives of this analysis include:

1. **Explore and preprocess customer interaction data** to ensure quality and consistency.
2. **Identify key factors and patterns** driving customer satisfaction.
3. **Analyze performance metrics** across different customer service teams.
4. **Develop predictive models** to forecast satisfaction levels and highlight areas of improvement.
5. **Recommend actionable strategies** to enhance overall customer service and CSAT scores.

By achieving these objectives, we can provide Flipkart with actionable insights that help improve customer satisfaction, agent performance, and service efficiency.

# **GitHub Link -**

https://github.com/kush-agra-soni/Flipkart_ML

# **Problem Statement**


Flipkart handles millions of customer interactions daily across multiple support channels, making it challenging to consistently deliver high-quality service. Delays in issue resolution, varying agent performance, and mismatched support strategies often result in lower CSAT scores and reduced customer trust.

This project aims to leverage machine learning to analyze customer interaction data, identify the factors influencing satisfaction, and predict CSAT outcomes. By uncovering patterns and actionable insights, the project provides Flipkart with data-driven strategies to optimize agent performance, resolve issues efficiently, and improve overall customer experience.

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import nltk
import shap
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PowerTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nltk.download('vader_lexicon')

### Dataset Loading

In [ ]:
# Load Dataset
url = "https://raw.githubusercontent.com/kush-agra-soni/Flipkart_ML/refs/heads/main/Customer_support_data.csv"
df = pd.read_csv(url)

### Dataset First View

In [ ]:
df.columns

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
df.shape

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_count = df.duplicated().sum()
print("Total duplicate rows:", duplicate_count)

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing_values = df.isnull().sum()

# Display columns with missing values only
print("Missing values per column:")
print(missing_values[missing_values > 0])

In [ ]:
# Visualizing the missing values
plt.figure(figsize=(15,3))
sns.barplot(x=missing_values.index, y=missing_values.values)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Missing Count")
plt.title("Missing Values per Column", fontsize=14)
plt.show()

### What did you know about your dataset?

The dataset contains **85,907 records** with **20 columns** capturing Flipkart customer support interactions.  
- **Target Variable:** `CSAT Score` (Customer Satisfaction Score).  
- **Data Types:**
  - Categorical: `channel_name`, `category`, `Sub-category`, `Agent_name`, `Supervisor`, etc.  
  - Numerical: `Item_price`, `connected_handling_time`, `CSAT Score`.  
  - Datetime: `order_date_time`, `Issue_reported at`, `Survey_response_Date`.  
- **Duplicates:** No duplicate rows found.  
- **Missing Values:**
  - High missingness in `connected_handling_time` (~99%).  
  - Significant missingness in `order_date_time`, `Customer_City`, `Product_category`, `Item_price`.  


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

In [ ]:
# Dataset Describe
df.describe()

### Variables Description

- **Unique id**: Unique identifier for each record.  
- **channel_name**: Customer support channel used (e.g., call, chat, email).  
- **category**: Broad classification of the issue reported.  
- **Sub-category**: More specific type of issue under the main category.  
- **Customer Remarks**: Free-text feedback or comments from customers.  
- **Order_id**: Associated order identifier (if applicable).  
- **order_date_time**: Date and time of the order placed.  
- **Issue_reported at**: Timestamp when the issue was first reported.  
- **issue_responded**: Timestamp or status when the issue was responded to.  
- **Survey_response_Date**: Date of customer survey response.  
- **Customer_City**: Location of the customer (if available).  
- **Product_category**: Type/category of product purchased.  
- **Item_price**: Price of the product involved in the interaction.  
- **connected_handling_time**: Handling time of the issue (mostly missing).  
- **Agent_name**: Name/identifier of the customer support agent.  
- **Supervisor**: Supervisor overseeing the support agent.  
- **Manager**: Manager responsible for the support team.  
- **Tenure Bucket**: Agent’s experience bucket (e.g., 0–6 months, 6–12 months).  
- **Agent Shift**: Shift during which the agent handled the issue.  
- **CSAT Score**: Customer Satisfaction Score (target variable for ML).  


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
unique_counts = df.nunique()

print("Unique Values per Column:\n")
print(unique_counts)

## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Drop redundant columns safely (no KeyError if already absent)
cols_to_drop = ['connected_handling_time', 'Customer_City', 'Unique id', 'Order_id', 'order_date_time', 'Survey_response_Date']
df = df.drop(columns=cols_to_drop, errors='ignore')

In [ ]:
# Convert with the correct format
df['Issue_reported at'] = df['Issue_reported at'].str.strip()
df['Issue_reported at'] = pd.to_datetime(df['Issue_reported at'], format='%d/%m/%Y %H:%M', errors='coerce')
df['issue_responded'] = pd.to_datetime(df['issue_responded'], format='%d/%m/%Y %H:%M')

In [ ]:
# Refining Customer Remarks Column
# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Function to convert text sentiment into 1–5 scale
def sentiment_to_score(text):
    if pd.isna(text):
        return np.nan
    score = sia.polarity_scores(str(text))['compound']  # range: -1 to +1
    if score <= -0.6:   # very negative
        return 1
    elif score <= -0.2: # negative
        return 2
    elif score <= 0.2:  # neutral
        return 3
    elif score <= 0.6:  # positive
        return 4
    else:               # very positive
        return 5

# Apply to dataset
df['customer_response'] = df['Customer Remarks'].apply(sentiment_to_score)

# Fill missing with median of available scores
median_score = df['customer_response'].median()
df['customer_response'] = df['customer_response'].fillna(median_score)

# Quick check
print(df['customer_response'].value_counts())

In [ ]:
# Plot sentiment distribution
plt.figure(figsize=(8,3))
df['customer_response'].value_counts().sort_index().plot(
    kind='bar',
    rot=0
)

plt.title("Customer Sentiment Distribution (1=Very Negative, 5=Very Positive)")
plt.xlabel("Sentiment Score")
plt.ylabel("Count")
plt.show()

In [ ]:
# Step 1: Impute missing Product_category
df['Product_category'] = df['Product_category'].fillna("Unknown")

# Step 2: Fill missing Item_price based on median per Product_category
df['Item_price'] = df.groupby('Product_category')['Item_price'].transform(
    lambda x: x.fillna(x.median())
)

# Step 3: If still missing (just in case), fill with overall median
df['Item_price'] = df['Item_price'].fillna(df['Item_price'].median())

# Verification
print("Missing values after category-based imputation:", df['Item_price'].isnull().sum())
print(df[['Product_category', 'Item_price']].tail(10))

In [ ]:
# Map Tenure Bucket to numeric ordinal values
tenure_mapping = {
    'On Job Training': 0,
    '0-30': 1,
    '31-60': 2,
    '61-90': 3,
    '>90': 4
}

# Create new numeric column
df['tenure_numeric'] = df['Tenure Bucket'].map(tenure_mapping)

# Quick check
print(df[['Tenure Bucket', 'tenure_numeric']].head(5))
print("Missing values in new column:", df['tenure_numeric'].isnull().sum())

In [ ]:
# Create average CSAT features for hierarchy

# Average CSAT per agent
df['agent_avg_csat'] = df.groupby('Agent_name')['CSAT Score'].transform('mean').round(2)

# Average CSAT per supervisor
df['supervisor_avg_csat'] = df.groupby('Supervisor')['CSAT Score'].transform('mean').round(2)

# Average CSAT per manager
df['manager_avg_csat'] = df.groupby('Manager')['CSAT Score'].transform('mean').round(2)

# Quick check
print(df[['Agent_name','agent_avg_csat','Supervisor','supervisor_avg_csat',
          'Manager','manager_avg_csat']].head())

In [ ]:
df.info()

### What all manipulations have you done and insights you found?

During the data wrangling phase, several key steps were performed to clean, transform, and prepare the Flipkart customer support dataset for analysis and ML modeling:

---

### 1️⃣ Dropping Redundant Columns
- Removed columns that were either highly sparse (`connected_handling_time`), identifiers (`Unique id`, `Order_id`), or redundant for ML (`order_date_time`, `Survey_response_Date`, `Customer_City`).
- This reduced noise and kept only actionable features for modeling.

---

### 2️⃣ Datetime Standardization
- Converted `Issue_reported at` and `issue_responded` into **consistent datetime format** (`DD-MM-YYYY HH:MM`).
- This allows calculation of response times, delays, and other temporal analyses.
- `order_date_time` and `Survey_response_Date` were dropped due to high missingness and low predictive value.

---

### 3️⃣ Customer Remarks Sentiment
- Applied **VADER sentiment analysis** to `Customer Remarks` to convert text into a **numeric scale (1–5)** for `customer_response`.
  - Negative → 1–2  
  - Neutral → 3  
  - Positive → 4–5
- Missing remarks were filled with the **median sentiment**.
- Insights:
  - Helps quantify customer mood.
  - Useful for ML models and understanding agent-level performance.

---

### 4️⃣ Product Category & Item Price Imputation
- Missing `Product_category` filled with `"Unknown"`.  
- `Item_price` missing values imputed using **median per product category**, then overall median as fallback.
- Insights:
  - Maintains price consistency across products.
  - Reduces bias due to missing values.

---

### 5️⃣ Tenure Bucket Transformation
- Converted `Tenure Bucket` into a numeric **ordinal feature** `tenure_numeric`:
  - `On Job Training` → 0, `0-30` → 1, `31-60` → 2, `61-90` → 3, `>90` → 4
- Insights:
  - Captures agent experience level as a numeric feature for ML.
  - Higher tenure may correlate with higher CSAT.

---

### 6️⃣ Hierarchical Aggregation Features
- Calculated **average CSAT scores** at different hierarchy levels:
  - `agent_avg_csat` → average CSAT per agent  
  - `supervisor_avg_csat` → average CSAT per supervisor  
  - `manager_avg_csat` → average CSAT per manager
- Insights:
  - Enables analysis of which agents, supervisors, or managers impact customer satisfaction.
  - Preserves hierarchical relationships in a compressed form for modeling.

---

### 7️⃣ Final Dataset Characteristics
- Total entries: 85,907  
- Total columns: 19  
- Key feature types:
  - 2 datetime columns
  - 10 categorical/object columns
  - 5 numeric/float columns
  - 2 integer columns
- All missing values in critical numeric and sentiment features have been addressed.  

---

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Sentiment distribution
plt.figure(figsize=(8,3))
sns.countplot(x='customer_response', data=df, palette="viridis", hue = 'customer_response', legend=False)
plt.title("Customer Sentiment (1=Very -, 5=Very +)")
plt.xlabel("Sentiment Score")
plt.ylabel("Count")
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart was chosen for this visualization because it is the most effective way to compare the counts or frequencies of different categorical variables. In this case, the sentiment scores (1.0, 2.0, 3.0, 4.0, 5.0) are distinct, categorical values. The bar chart's height directly and intuitively represents the count of customers for each score, making it easy to see which sentiment score is the most and least frequent. It allows for a straightforward comparison of the number of customers who gave a 4-star rating versus a 1-star rating, for example, which is a key objective of this analysis. The clear separation of the bars prevents any misinterpretation that might occur with a continuous-scale chart, and its simplicity makes the insight immediately apparent.

##### 2. What is/are the insight(s) found from the chart?

The primary insight from this chart is that Flipkart customers are overwhelmingly satisfied. The vast majority of customer ratings fall into the positive categories, specifically a score of 4.0. The highest bar by a significant margin is at 4.0, indicating that a large number of customers are highly satisfied but not ecstatic. While the number of customers who gave a perfect 5.0 is much lower than those who gave a 4.0, it is still the second-highest category, reinforcing the positive trend. Conversely, the number of negative (1.0 and 2.0) and neutral (3.0) reviews are very low. This suggests that while there are some dissatisfied customers, they represent a small minority of the total customer base.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can be leveraged to create a positive business impact. The data clearly indicates that Flipkart has a strong foundation of satisfied customers. This information can be used to validate existing strategies and provide a basis for future growth. For example, marketing teams can use the high sentiment scores as a strong selling point in their campaigns, highlighting customer satisfaction. The operations team can focus on the small segment of dissatisfied customers to understand the root causes of their negative sentiment, such as delivery issues or product quality, and implement targeted improvements. By understanding what is working (leading to the high number of 4.0 ratings) and addressing the few areas of weakness, the business can move more customers from a 4.0 to a 5.0 rating and proactively prevent negative experiences, thereby improving overall customer loyalty and lifetime value.

#### Chart - 2

In [ ]:
# Top 10 categories by volume
plt.figure(figsize=(8,3))
df['category'].value_counts().head(10).plot(kind='bar')
plt.title("Top 10 Categories by Case Volume")
plt.xlabel("Category")
plt.ylabel("Count")
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart is the ideal choice here because it effectively displays the Top 10 categories of customer issues by case volume. The categorical nature of the data, with distinct labels like 'Returns' and 'Order Related', is perfectly suited for a bar chart. The varying heights of the bars make it easy to compare the frequency of each category at a glance, immediately highlighting which issues are most common and which are least. This visual representation allows for quick identification of the biggest problem areas.

##### 2. What is/are the insight(s) found from the chart?

The primary insight is that "Returns" and "Order Related" issues are the most significant pain points for Flipkart customers, accounting for the vast majority of support cases. "Returns" alone has over 40,000 cases, which is nearly double the volume of the next highest category, "Order Related." This indicates that customers are frequently dissatisfied with products they receive and the overall order fulfillment process. Other categories like "Refund Related" and "Product Queries" are far less frequent, suggesting that issues with payment and product information are not as common.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can drive a substantial positive business impact. By identifying "Returns" and "Order Related" issues as the leading causes of customer support cases, Flipkart can prioritize efforts to address these specific problems. The company can investigate the root causes of returns (e.g., product quality, inaccurate descriptions, damaged goods) and order issues (e.g., late deliveries, incorrect items, tracking problems). By implementing targeted solutions—such as improving quality control, optimizing logistics, or enhancing product descriptions—they can reduce the volume of these cases. This reduction would lead to lower operational costs, a more efficient support system, and, most importantly, higher customer satisfaction and loyalty

#### Chart - 3

In [ ]:
# CSAT distribution
plt.figure(figsize=(5,3))
sns.histplot(df['CSAT Score'], bins=5, kde=False)
plt.title("CSAT Score Distribution")
plt.xlabel("CSAT Score")
plt.ylabel("Count")
plt.show()

##### 1. Why did you pick the specific chart?

A histogram is the appropriate choice for this visualization because it shows the distribution of a continuous variable, which is the CSAT (Customer Satisfaction) score in this case. By grouping scores into bins (e.g., 1.0-1.5, 1.5-2.0, etc.), the histogram reveals the frequency of scores within specific ranges. This allows for a clear understanding of the overall shape and spread of the data, immediately showing where the bulk of customer satisfaction scores lie, and highlighting the dominance of high scores

##### 2. What is/are the insight(s) found from the chart?

The key insight is that the CSAT scores are heavily skewed towards the positive end. The tallest bar, by a significant margin, is for scores between 4.0 and 5.0, indicating that the vast majority of customers are highly satisfied. There is a smaller peak for scores between 1.0 and 1.5, suggesting a group of highly dissatisfied customers, but they are a small minority. The low frequency of scores in the middle range (2.0-4.0) indicates that customers tend to be either very happy or very unhappy, with few neutral or moderately negative opinions

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can create a positive business impact. The strong positive skew of the data provides confidence that current customer satisfaction strategies are working. The company can leverage this high satisfaction rate in marketing and public relations. Furthermore, the small but distinct group of low scores (1.0-1.5) provides a clear target for intervention. By analyzing the common issues of these highly dissatisfied customers, Flipkart can implement targeted improvements to prevent negative experiences, which in turn can reduce churn and improve overall brand reputation.

#### Chart - 4

In [ ]:
# Top 10 agents with lowest avg CSAT
low_agents = df.groupby('Agent_name')['agent_avg_csat'].mean().sort_values().head(10)

plt.figure(figsize=(9,3))
low_agents.plot(kind='bar', color='red')
plt.title("Bottom 10 Agents by Avg CSAT")
plt.ylabel("Avg CSAT")
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart is best for this data because it allows for a clear, direct comparison of average CSAT scores among the bottom 10 agents. By assigning each agent a separate bar, the chart visually ranks their performance, making it simple to identify who has the lowest and highest average score within this group. This visual ranking helps in easily pinpointing the agents who need the most immediate attention and support.

##### 2. What is/are the insight(s) found from the chart?

The key insight is the low average CSAT scores across all agents in this group. All of the bottom 10 agents have average scores below 2.6, which is a very low satisfaction rating. The chart shows a slight upward trend, from Philip Harmon with the lowest score to Veronica Anderson with the highest among the bottom 10, but all are far from a satisfactory level. This indicates a systemic issue with a portion of the support team rather than an isolated problem with one or two individuals.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can have a significant positive business impact. By identifying the lowest-performing agents, Flipkart can take targeted action. This isn't just about punishment; it's about implementing training, providing coaching, and addressing potential issues with their tools or workload. Improving the performance of these agents could directly lead to a reduction in customer dissatisfaction and a better overall CSAT score. This focused effort is a cost-effective way to improve a key business metric.

#### Chart - 5

In [ ]:
# Average CSAT by channel
plt.figure(figsize=(6,3))
sns.barplot(x='channel_name', y='Item_price', data=df, estimator=np.mean, errorbar=None)
plt.title("Average Item price by Channel")
plt.xticks(rotation=45)
plt.show()

##### 1. Why did you pick the specific chart?

A bar chart is the ideal choice as it provides a simple, direct comparison of the average item price across different communication channels. The distinct categories on the x-axis (Outcall, Inbound, Email) make a bar chart the most effective way to visually represent and compare the average price associated with each channel.

##### 2. What is/are the insight(s) found from the chart?

The key insight is that customers using email as a support channel tend to have purchased higher-priced items than those who use inbound or outbound calls. The average item price for email-based support is significantly higher, at over ₹3000, while the average for calls is much lower. This suggests a correlation between the value of a customer's purchase and their preferred method of seeking support.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes, these insights can create a positive business impact. By knowing that higher-value customers often use email support, Flipkart can prioritize and enhance their email support system. This could involve assigning more experienced agents to email tickets, developing specialized workflows for high-value products, or ensuring faster response times.

## ***5. Hypothesis Testing***

### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Agent effect (from Bottom 10 Agents by Avg CSAT)

- H₀: Mean CSAT of the bottom-10 agents = mean CSAT of all other agents.
- H₁: Mean CSAT of the bottom-10 agents < mean CSAT of all other agents.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Perform Statistical Test to obtain P-Value
# Hypothesis 1: bottom-10 agents vs others (one-sided)
from scipy.stats import mannwhitneyu
alpha = 0.05

# find bottom 10 agents by average CSAT
agent_mean = df.groupby('Agent_name')['CSAT Score'].mean()
bottom10 = agent_mean.nsmallest(10).index.tolist()

csat_bottom = df[df['Agent_name'].isin(bottom10)]['CSAT Score']
csat_others = df[~df['Agent_name'].isin(bottom10)]['CSAT Score']

print("N bottom10:", len(csat_bottom), "N others:", len(csat_others))
print("Median bottom10:", csat_bottom.median(), "Median others:", csat_others.median())

stat, p = mannwhitneyu(csat_bottom, csat_others, alternative='less')
print("Mann-Whitney U stat:", stat, "p-value:", p)
if p < alpha:
    print("Conclusion: Reject H0 — bottom-10 agents have significantly LOWER CSAT (one-sided).")
else:
    print("Conclusion: Fail to reject H0 — no evidence bottom-10 are lower.")

##### Which statistical test have you done to obtain P-Value?

Mann–Whitney U (non-parametric two-sample test, one-sided less).

##### Why did you choose the specific statistical test?

CSAT is ordinal/discrete and groups may be non-normal and unequal variance — Mann–Whitney is robust.

### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Channel differences (from Average Item price by Channel & channel analyses)

- H₀: CSAT distributions are the same across channels.
- H₁: At least one channel has a different CSAT distribution.

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 2: Channel vs CSAT (Kruskal-Wallis)
from scipy.stats import kruskal
alpha = 0.05

groups = [grp['CSAT Score'].values for name, grp in df.groupby('channel_name')]
names = df['channel_name'].unique().tolist()
print("Channels tested:", names)

hstat, p = kruskal(*groups)
print("Kruskal-Wallis H-stat:", hstat, "p-value:", p)
if p < alpha:
    print("Conclusion: Reject H0 — CSAT differs across channels (at least one channel differs).")
else:
    print("Conclusion: Fail to reject H0 — no evidence of differences across channels.")

##### Which statistical test have you done to obtain P-Value?

Kruskal–Wallis H-test (non-parametric ANOVA).

##### Why did you choose the specific statistical test?

More than two groups (channels). CSAT is ordinal/non-normal → Kruskal–Wallis is appropriate.

### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

Tenure effect (from Tenure vs performance idea)

- H₀: No monotonic association between agent tenure and CSAT.
- H₁: There is a monotonic association (higher tenure → different CSAT).

#### 2. Perform an appropriate statistical test.

In [ ]:
# Hypothesis 3: Tenure vs CSAT (Spearman)
from scipy.stats import spearmanr
alpha = 0.05

corr, p = spearmanr(df['tenure_numeric'], df['CSAT Score'], nan_policy='omit')
print("Spearman rho:", corr, "p-value:", p)
if p < alpha:
    print("Conclusion: Reject H0 — significant monotonic association between tenure and CSAT.")
    if corr > 0:
        print("Direction: Positive association (higher tenure -> higher CSAT).")
    else:
        print("Direction: Negative association (higher tenure -> lower CSAT).")
else:
    print("Conclusion: Fail to reject H0 — no evidence of monotonic association.")

##### Which statistical test have you done to obtain P-Value?

Spearman rank correlation

##### Why did you choose the specific statistical test?

**tenure_numeric** is ordinal and CSAT is ordinal — Spearman measures monotonic relationships without normality assumptions.

## ***6. Feature Engineering & Data Pre-processing***

### A. Categorical Encoding

In [ ]:
# Encode your categorical columns
# Drop unnecessary categorical text columns
df = df.drop(columns=['Customer Remarks', 'Tenure Bucket'])

# One-Hot Encoding for low-cardinality categoricals
low_card = ['channel_name', 'category', 'Agent Shift']
df = pd.get_dummies(df, columns=low_card, drop_first=True)

# Target encoding for medium-cardinality
# Example: Sub-category, Product_category
for col in ['Sub-category', 'Product_category']:
    mean_map = df.groupby(col)['CSAT Score'].mean()
    df[col + '_mean_csat'] = df[col].map(mean_map)

# Drop the original high-cardinality IDs (Agent_name, Supervisor, Manager)
df = df.drop(columns=['Agent_name', 'Supervisor', 'Manager'])

In [ ]:
# Drop raw categorical after encoding
df = df.drop(columns=['Sub-category', 'Product_category'])

# Extract datetime features (example: day, month, hour, response time already exists)
df['issue_reported_day'] = df['Issue_reported at'].dt.day
df['issue_reported_month'] = df['Issue_reported at'].dt.month

df['issue_responded_day'] = df['issue_responded'].dt.day
df['issue_responded_month'] = df['issue_responded'].dt.month

# Drop original datetime columns if only response_time_mins is needed
df = df.drop(columns=['Issue_reported at', 'issue_responded'])

#### What all categorical encoding techniques have you used & why did you use those techniques?

We have to deal with **encode categorical variables**, **extract useful time-based features** and **add meaningful numerical features** without bloating the dataset.

---

### 🔹 Steps Performed

1. **Dropped Non-Useful Columns**

   * Removed high-cardinality identifiers like `Agent_name`, `Supervisor`, and `Manager`.
   * Replaced them with aggregated features (`*_avg_csat`) for hierarchical performance analysis.

2. **Categorical Encoding**

   * Applied **one-hot encoding** on categorical variables:

     * `channel_name`
     * `category`
     * `Agent Shift`
   * Result: new binary columns (dtype = `bool`) that are memory-efficient and ML-ready.

3. **Target Encoding for Categories**

   * Created category-level CSAT performance features:

     * `Sub-category_mean_csat`
     * `Product_category_mean_csat`
   * Captures customer satisfaction signal of categories without creating too many columns.

4. **Datetime Feature Engineering**

   * Extracted useful components from `Issue_reported at` and `issue_responded`:

     * `day`, `month`, `hour`
   * Helps models capture **time-of-day** and **seasonality effects** on customer satisfaction.

5. **Numeric & Sentiment Features**

   * Already engineered in wrangling phase:

     * `customer_response` → sentiment score (1–5) from remarks.
     * `tenure_numeric` → ordinal mapping of tenure bucket.
     * `agent_avg_csat`, `supervisor_avg_csat`, `manager_avg_csat` → hierarchy-based averages.

---

### 🔹 Final Dataset Structure

* **Rows**: 85,907
* **Columns**: 32 (mix of numerical + boolean features)
* **Data Types**:

  * `float64`: continuous features (`Item_price`, CSAT averages, sentiment)
  * `int64` / `int32`: ordinal features (`tenure_numeric`, day, month, hour)
  * `bool`: one-hot encoded categorical features

---

### B. Feature Manipulation & Selection

In [ ]:
# Select your features wisely to avoid overfitting
# Compute correlation matrix
corr_matrix = df.corr()  # only numeric/bool columns are considered automatically

# Extract correlation with target
csat_corr = corr_matrix['CSAT Score'].sort_values(ascending=False)
print("Correlation with CSAT Score:\n", csat_corr)

In [ ]:
# Generate correlation table for top features
# Take absolute correlation for ranking
top_features = csat_corr.drop('CSAT Score').abs().sort_values(ascending=False).head(10)
print("\nTop 10 features affecting CSAT Score:\n", top_features)

In [ ]:
# Correlation heatmap for top features
plt.figure(figsize=(12,6))
sns.heatmap(df[top_features.index.tolist() + ['CSAT Score']].corr(),
            annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation of Top Features with CSAT Score")
plt.show()

In [ ]:
# Optional: Bar plot of correlations
plt.figure(figsize=(8,3))
top_features.plot(kind='bar', color='skyblue')
plt.title("Top 10 Features Correlated with CSAT Score")
plt.ylabel("Correlation (absolute)")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
top_10_features = [
    'customer_response',
    'agent_avg_csat',
    'Sub-category_mean_csat',
    'Product_category_mean_csat',
    'Item_price',
    'supervisor_avg_csat',
    'category_Returns',
    'category_Order Related',
    'manager_avg_csat',
    'tenure_numeric'
]

df = df[top_10_features + ['CSAT Score']]

##### What all feature selection methods have you used  and why?

1. **Domain Knowledge:**  
   - Dropped irrelevant or redundant columns (`Unique id`, `Order_id`, `Customer Remarks`) before encoding.  
   - Focused on features logically impacting CSAT, such as price, agent hierarchy, tenure, category, and time.

2. **Correlation Analysis:**  
   - Computed Pearson correlation between all numeric/boolean features and `CSAT Score`.  
   - Identified features with strongest linear relationships to the target, helping prioritize features for modeling.

3. **Hierarchical & Aggregated Features:**  
   - Engineered features like `agent_avg_csat`, `supervisor_avg_csat`, `manager_avg_csat`, `Sub-category_mean_csat`, and `Product_category_mean_csat` to summarize historical satisfaction patterns and reduce dimensionality.

4. **One-Hot Encoding & Avoiding Multicollinearity:**  
   - Converted categorical variables into numeric form using one-hot encoding.  
   - Avoided using both raw categories and their aggregated CSAT features simultaneously in some models to reduce multicollinearity.

---

##### Which all features you found important and why?

**Top 10 features affecting CSAT Score:**

1. `customer_response` (0.341) – Derived from sentiment analysis on customer remarks; strongest predictor of satisfaction.  
2. `agent_avg_csat` (0.247) – Reflects individual agent performance historically.  
3. `Sub-category_mean_csat` (0.175) – Captures satisfaction trends at the sub-category level.  
4. `Product_category_mean_csat` (0.113) – Indicates product-level satisfaction differences.  
5. `Item_price` (0.091) – Product value can influence customer expectations and CSAT.  
6. `supervisor_avg_csat` (0.083) – Performance of supervising staff impacts overall CSAT.  
7. `category_Returns` (0.078) – Return-related interactions have measurable effect on satisfaction.  
8. `category_Order Related` (0.064) – Order-related queries slightly influence CSAT.  
9. `manager_avg_csat` (0.061) – Manager-level oversight contributes to agent performance.  
10. `tenure_numeric` (0.037) – Agent experience moderately affects satisfaction outcomes.

**Insights:**  
- Sentiment-based features are the strongest drivers.  
- Hierarchical performance metrics show both individual and team-level impact.  
- Category-specific and transactional features provide additional contextual relevance.  
- These features form the core set for predictive modeling and analysis.

### C. Data Transformation

In [ ]:
# Define the top 10 important feature names
top_features = ['customer_response','agent_avg_csat','Sub-category_mean_csat','Product_category_mean_csat',
                'Item_price','supervisor_avg_csat', 'category_Returns','category_Order Related','manager_avg_csat','tenure_numeric']

# Plot histograms only for those top features
df[top_features].hist(bins=50, figsize=(12, 9), edgecolor='black')
plt.suptitle('Histograms of Top 10 Features Affecting CSAT Score', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to make room for title
plt.show()

In [ ]:
# Features to transform
features = ['customer_response', 'agent_avg_csat',
            'Sub-category_mean_csat', 'Product_category_mean_csat',
            'supervisor_avg_csat', 'Item_price']

# Use Yeo-Johnson (handles both positive/negative skew)
pt = PowerTransformer(method='yeo-johnson', standardize=False)

# Transform features (replace old ones directly)
df[features] = pt.fit_transform(df[features])

# Print skewness before and after
print("Skewness comparison:")
for col in features:
    print(f"{col}: transformed skew = {df[col].skew():.3f}")

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

Yes, data transformation was required because several numerical features in the dataset showed **high skewness** (both left and right).  
Skewed features can negatively impact models like **linear regression, logistic regression, and neural networks** which assume that input features follow approximately normal distributions.  

#### Transformation Used:
- I applied **Yeo–Johnson Transformation** (via `PowerTransformer` from `sklearn`).  
- Yeo–Johnson is preferred because:
  - It handles **both positive and negative values** (unlike log which requires strictly positive values).  
  - It works for **right-skewed and left-skewed distributions** simultaneously.  
  - It reduces skewness, bringing features closer to a **normal (Gaussian-like) distribution**.  

#### Outcome:
- After transformation, **5 out of 6 skewed features** fell within the acceptable skewness range (-0.5 to +0.5), making them much more symmetric and suitable for modeling.  
- Only one feature (`Product_category_mean_csat`) remained moderately skewed, but it is still usable (especially for tree-based models that are robust to skewness).  

### D. Data Scaling

In [ ]:
df.info()

In [ ]:
# Scaling your data
# Select numeric columns (skip bools + exclude target column)
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
target_col = 'CSAT Score'
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

# Initialize scaler
scaler = StandardScaler()

# Fit + transform only features (not target)
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

print("Scaled dataset preview:")
print(df.head())

##### Which method have you used to scale you data and why?

* **Consistency across features:** My dataset contains features on very different scales (e.g., `Item_price` vs. `CSAT related scores`). Standardization brings them to a common scale with **mean = 0** and **standard deviation = 1**.
* **Handles outliers better than MinMax:** Unlike Min-Max scaling, which compresses all values into the range `[0,1]`, standardization is less sensitive to outliers and does not squash the majority of data into a narrow range.
* **Suitable for ML algorithms:** Many machine learning algorithms (e.g., Logistic Regression, SVM, Neural Networks, Gradient Descent–based models) perform better and converge faster when features are standardized.
* **Preserved Target Variable:** The scaling was applied only to the input features. The target variable (`CSAT Score`) was left untouched to maintain its interpretability.

### F. Data Splitting

In [ ]:
# Separate features and target
X = df.drop(columns=['CSAT Score'])
y = df['CSAT Score']

# First split: train vs temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Second split: validation vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("Shapes after splitting:")
print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
def check_distribution(y, name):
    print(f"\n{name} Distribution:")
    print(y.value_counts(normalize=True).sort_index())

# Check distributions
check_distribution(y_train, "Train")
check_distribution(y_val, "Validation")
check_distribution(y_test, "Test")

In [ ]:
plt.figure(figsize=(8,4))
plt.hist([y_train, y_val, y_test], bins=len(y.unique()), label=['Train', 'Val', 'Test'], stacked=True)
plt.xlabel("CSAT Score")
plt.ylabel("Count")
plt.title("Class Distribution Across Splits")
plt.legend()
plt.show()

##### What data splitting ratio have you used and why?

1. **Training Set (70%)**

   * Contains the majority of the data to allow the model to learn general patterns.
   * Ensures enough examples from minority classes to reduce bias in predictions.

2. **Validation Set (15%)**

   * Used for hyperparameter tuning and model selection.
   * Provides an unbiased check on model performance during training.

3. **Test Set (15%)**

   * Fully unseen data to evaluate final model performance.
   * Ensures the reported accuracy or metrics are realistic and generalizable.

**Why this ratio:**

* Large enough **train set** ensures stable learning.
* Adequate **validation and test sets** give reliable estimates of performance.
* Common practice for datasets of moderate size (\~85k rows) to avoid overfitting while maintaining representativeness of all classes.

**Additional Note:**

* **Stratified splitting** was applied to maintain the same class distribution of `CSAT Score` across all splits, which is crucial for highly imbalanced target variables.


## ***7. ML Model Implementation***

### ML Model - 1

In [ ]:
# Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    random_state=42,
    class_weight="balanced"  # handles imbalance
)
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Visualizing evaluation Metric Score chart
print("Random Forest Report:")
print(classification_report(y_test, rf_pred))

**Model Used:** Random Forest Classifier  
- Ensemble model using 200 trees (`n_estimators=200`)
- `class_weight="balanced"` to handle class imbalance

**Performance Overview:**
- **Accuracy:** 54%
- **Class 5 (CSAT = 5)**: High performance (Precision: 0.73, Recall: 0.70)
- **Classes 2 & 3**: Very poor performance (F1-scores ≈ 0.03–0.04)
- **Macro F1-Score:** 0.25 → Low overall performance across all classes

**Key Issue:**  
- Severe **class imbalance** → model biased towards class 5


In [ ]:
# Plotting for evaluation
cm_rf = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(5,3))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### ML Model - 2

In [ ]:
#2. XGBoost Classifier

# Shift labels from [1-5] to [0-4]
y_train_xgb = y_train - 1
y_val_xgb   = y_val - 1
y_test_xgb  = y_test - 1

# XGBoost Model
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=5,               # CSAT has 5 categories
    random_state=42
)

xgb_model.fit(X_train, y_train_xgb)

# Predictions (in [0-4])
xgb_pred = xgb_model.predict(X_test)

# Shift back to [1-5]
xgb_pred = xgb_pred + 1

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
print("XGBoost Report:")
print(classification_report(y_test, xgb_pred))

In [ ]:
cm_xgb = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(6,4))
sns.heatmap(cm_xgb, annot=True, fmt="d", cmap="Greens")
plt.title("Confusion Matrix - XGBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Evaluation Matrix:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision (macro)": precision_score(y_true, y_pred, average="macro"),
        "Recall (macro)": recall_score(y_true, y_pred, average="macro"),
        "F1-score (macro)": f1_score(y_true, y_pred, average="macro")
    }

results = []
results.append(evaluate_model("Random Forest", y_test, rf_pred))
results.append(evaluate_model("XGBoost", y_test, xgb_pred))

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df)


### 1. Which Evaluation metrics did you consider for a positive business impact and why?

### Evaluation Metrics for Positive Business Impact

For this multi-class CSAT prediction model, we considered the following metrics:

#### 1. **F1-Score (Macro & Weighted)**
- **Why:** F1-score balances **precision** (how many predicted positives were correct) and **recall** (how many actual positives were captured).
- **Macro F1** treats all classes equally → important when minority classes (e.g., CSAT 2 or 3) are critical for early dissatisfaction detection.
- **Weighted F1** accounts for class imbalance → reflects overall performance realistically.

#### 2. **Recall (Especially for Low CSAT Classes)**
- **Why:** High recall for **low CSAT scores (1, 2, 3)** is critical to identify **unhappy customers** early and trigger corrective actions.
- Missing these leads to **customer churn** and **negative word-of-mouth**, both harmful to business.

#### 3. **Precision for High CSAT (Score = 5)**
- **Why:** Helps measure the **reliability** of identifying highly satisfied customers.
- Important for **upselling**, **loyalty programs**, and **customer referrals**.

#### 4. **Accuracy**
- Useful as an overall measure, but **not sufficient alone** due to class imbalance (e.g., CSAT = 5 dominates).

---

### Business Impact:
- **Early detection of dissatisfaction (via recall for low scores)** → reduced churn, better CX.
- **Accurate identification of promoters (CSAT = 5)** → improved retention and monetization strategies.
- **Balanced F1-scores** → model performs fairly across all customer segments, not just the majority

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

### Final Model Selection & Justification

**Chosen Model:** **Random Forest Classifier with Hyperparameter Optimization**

#### Why this Model?
- **Best Balance of Performance Metrics:**
  - After tuning via **RandomizedSearchCV**, the Random Forest model showed improved **macro & weighted F1-scores**, especially for minority CSAT classes.
  - Delivered better **recall for low CSAT scores (1, 2, 3)**, which is critical for identifying dissatisfied customers early.

- **Handles Class Imbalance Well:**
  - With `class_weight="balanced"`, the model could better learn from underrepresented CSAT scores.

- **Interpretability & Feature Importance:**
  - Random Forest provides insights into **which features impact CSAT**, helping the business take actionable steps.

- **Robust to Overfitting:**
  - Compared to individual decision trees, Random Forest generalizes better on unseen data.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

### 3. Model Explanation & Feature Importance Using SHAP

#### Model Used:
We used a **Random Forest Classifier** as our final model — an ensemble of decision trees that improves performance by averaging the predictions from multiple trees.

- It handles both **non-linear relationships** and **feature interactions** well.
- With `class_weight="balanced"`, it addresses **class imbalance** in CSAT scores.
- It’s also **robust**, interpretable, and requires minimal preprocessing.

---

# **Conclusion**

In this project, we built and fine-tuned a Random Forest Classifier to predict Customer Satisfaction (CSAT) scores using a rich feature set related to agent performance, product categories, and customer behavior. After hyperparameter optimization, the model showed improved recall and F1-scores, especially for identifying dissatisfied customers — a key factor for reducing churn and improving customer experience. Using SHAP, we interpreted the model’s predictions and identified important drivers of satisfaction, enabling actionable business insights. Overall, the model balances predictive performance and interpretability, making it suitable for real-world deployment in customer support analytics.
